# Automated RAG Evaluation

This notebook evaluates the frozen development-set RAG outputs using Ragas, a recognised automated evaluation framework.

## Objectives

- Reuse the 12 generation-evaluation cases frozen in Notebook 07.
- Evaluate the existing question, retrieved context, and final generated response without regenerating workflow outputs.
- Measure automated RAG quality using:
  - faithfulness,
  - answer relevancy,
  - context precision,
  - context recall.
- Compare automated findings with the documented human-review evidence.
- Identify low-scoring cases and likely retrieval or generation failure patterns.
- Export reproducible per-case, summary, failure-analysis, and manifest artifacts.

## Evaluation boundary

This notebook:

- uses development evaluation cases only,
- does not access held-out claims or held-out labels,
- does not rerun the LangGraph workflow,
- does not regenerate answers,
- does not rebuild the FAISS index,
- does not modify retrieval, prompts, deterministic rules, or guardrails.

Deterministic claim outcomes remain authoritative. Ragas scores assess the quality of the non-authoritative RAG-assisted analyst guidance only.

## 1. Environment and Repository Setup

Confirm that Notebook 09 is running in the isolated Ragas evaluation environment and resolve the repository root.

In [1]:
from pathlib import Path
import ast
import json
import os
import sys

import pandas as pd
import ragas


def find_project_root(start_path: Path) -> Path:
    """Locate the repository root using the expected project folders."""
    current_path = start_path.resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing src, data, and notebooks."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Ragas version:", ragas.__version__)

assert "dpclaims-ragas" in sys.executable, (
    "Notebook 09 must run in the isolated dpclaims-ragas environment."
)

assert ragas.__version__ == "0.3.9"

print("PASS: isolated Ragas environment confirmed")

/opt/anaconda3/envs/dpclaims-ragas/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
Python executable: /opt/anaconda3/envs/dpclaims-ragas/bin/python
Python version: 3.11.15
Ragas version: 0.3.9
PASS: isolated Ragas environment confirmed


## 2. Frozen Evaluation Inputs

Load the committed 12-case generation artifact and the completed human-review artifact.

The full generation artifact is used as the source because it preserves the complete retrieved-guidance JSON needed to reconstruct individual context chunks.

In [3]:
GENERATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "generation"
)

ARTIFACT_PATHS = {
    "generation_cases": (
        GENERATION_DIR
        / "generation_evaluation_cases_v1.csv"
    ),
    "human_review": (
        GENERATION_DIR
        / "generation_human_review_v2.csv"
    ),
    "judge_input": (
        GENERATION_DIR
        / "generation_judge_input_v2.csv"
    ),
    "judge_results": (
        GENERATION_DIR
        / "generation_llm_judge_results_v2.csv"
    ),
    "calibration_summary": (
        GENERATION_DIR
        / "generation_calibration_summary_v2.csv"
    ),
}

for artifact_name, artifact_path in ARTIFACT_PATHS.items():
    assert artifact_path.exists(), (
        f"Missing required artifact: {artifact_name} -> {artifact_path}"
    )

    print(
        "PASS:",
        artifact_name,
        "->",
        artifact_path.relative_to(PROJECT_ROOT),
    )

PASS: generation_cases -> data/evaluation/generation/generation_evaluation_cases_v1.csv
PASS: human_review -> data/evaluation/generation/generation_human_review_v2.csv
PASS: judge_input -> data/evaluation/generation/generation_judge_input_v2.csv
PASS: judge_results -> data/evaluation/generation/generation_llm_judge_results_v2.csv
PASS: calibration_summary -> data/evaluation/generation/generation_calibration_summary_v2.csv


In [4]:
generation_cases = pd.read_csv(
    ARTIFACT_PATHS["generation_cases"]
)

human_review = pd.read_csv(
    ARTIFACT_PATHS["human_review"]
)

judge_input = pd.read_csv(
    ARTIFACT_PATHS["judge_input"]
)

judge_results = pd.read_csv(
    ARTIFACT_PATHS["judge_results"]
)

calibration_summary = pd.read_csv(
    ARTIFACT_PATHS["calibration_summary"]
)

print("Generation cases:", generation_cases.shape)
print("Human review:", human_review.shape)
print("Judge input:", judge_input.shape)
print("Judge results:", judge_results.shape)
print("Calibration summary:", calibration_summary.shape)

Generation cases: (12, 66)
Human review: (12, 41)
Judge input: (12, 66)
Judge results: (12, 15)
Calibration summary: (4, 7)


## 3. Validate the Frozen Development Boundary

Confirm that all evaluation artifacts refer to the same 12 development cases and that the fields required for automated RAG evaluation are present.

In [5]:
EXPECTED_CASE_COUNT = 12

REQUIRED_GENERATION_COLUMNS = {
    "evaluation_case_id",
    "claim_id",
    "deterministic_outcome",
    "triggering_rule_id",
    "precedence_stage",
    "deterministic_reason",
    "controlled_query_text",
    "retrieval_status",
    "retrieved_guidance_count",
    "retrieved_chunk_ids",
    "retrieved_context",
    "retrieved_guidance_json",
    "reranking_status",
    "reranker_model_name",
    "final_outcome",
    "final_triggering_rule_id",
    "final_explanation",
    "content_safety_status",
    "authority_guardrail_status",
}

REQUIRED_HUMAN_COLUMNS = {
    "evaluation_case_id",
    "claim_id",
    "context_relevance_human",
    "groundedness_human",
    "answer_relevance_human",
    "hallucination_control_human",
    "critical_safety_failure_human",
    "human_notes",
}

missing_generation_columns = (
    REQUIRED_GENERATION_COLUMNS
    - set(generation_cases.columns)
)

missing_human_columns = (
    REQUIRED_HUMAN_COLUMNS
    - set(human_review.columns)
)

assert not missing_generation_columns, (
    "Generation cases missing columns: "
    + ", ".join(sorted(missing_generation_columns))
)

assert not missing_human_columns, (
    "Human review missing columns: "
    + ", ".join(sorted(missing_human_columns))
)

generation_case_ids = set(
    generation_cases["evaluation_case_id"]
)

human_case_ids = set(
    human_review["evaluation_case_id"]
)

judge_input_case_ids = set(
    judge_input["evaluation_case_id"]
)

judge_result_case_ids = set(
    judge_results["evaluation_case_id"]
)

assert len(generation_cases) == EXPECTED_CASE_COUNT
assert generation_cases["evaluation_case_id"].nunique() == EXPECTED_CASE_COUNT
assert generation_cases["claim_id"].nunique() == EXPECTED_CASE_COUNT

assert len(human_review) == EXPECTED_CASE_COUNT
assert human_review["evaluation_case_id"].nunique() == EXPECTED_CASE_COUNT

assert generation_case_ids == human_case_ids
assert generation_case_ids == judge_input_case_ids
assert generation_case_ids == judge_result_case_ids

assert generation_cases["workflow_status"].eq("COMPLETED").all()
assert generation_cases["retrieval_status"].eq("RESULTS_FOUND").all()
assert generation_cases["retrieved_guidance_count"].eq(3).all()

assert generation_cases["final_outcome"].eq(
    generation_cases["deterministic_outcome"]
).all()

assert generation_cases["final_triggering_rule_id"].eq(
    generation_cases["triggering_rule_id"]
).all()

assert generation_cases["content_safety_status"].eq("SAFE").all()
assert generation_cases["authority_guardrail_status"].eq("ALIGNED").all()

print("PASS: frozen generation cases =", EXPECTED_CASE_COUNT)
print("PASS: generation, human, and judge case IDs match")
print("PASS: all workflow executions completed")
print("PASS: all cases contain three retrieved guidance chunks")
print("PASS: final outcomes preserve deterministic outcomes")
print("PASS: final rules preserve deterministic rules")
print("PASS: all content-safety checks are SAFE")
print("PASS: all authority guardrails are ALIGNED")
print("PASS: no held-out artifact was loaded")

PASS: frozen generation cases = 12
PASS: generation, human, and judge case IDs match
PASS: all workflow executions completed
PASS: all cases contain three retrieved guidance chunks
PASS: final outcomes preserve deterministic outcomes
PASS: final rules preserve deterministic rules
PASS: all content-safety checks are SAFE
PASS: all authority guardrails are ALIGNED
PASS: no held-out artifact was loaded


## 4. Reconstruct Ragas Evaluation Fields

Ragas requires four core fields:

- **user input**: the controlled analyst-guidance query,
- **retrieved contexts**: the three individual retrieved KB chunks,
- **response**: the frozen final generated explanation,
- **reference**: an authoritative expected answer constructed only from deterministic facts.

The reference answer is not copied from the generated response. It is built from the deterministic outcome, triggering rule, precedence stage, and deterministic reason so that the evaluated answer is not used as its own ground truth.

In [6]:
def parse_json_list(value: object, field_name: str) -> list:
    """Parse and validate a JSON-encoded list from the committed artifact."""
    if pd.isna(value):
        raise ValueError(f"{field_name} is missing.")

    parsed_value = json.loads(str(value))

    if not isinstance(parsed_value, list):
        raise ValueError(
            f"{field_name} must contain a JSON list."
        )

    return parsed_value


def extract_retrieved_contexts(value: object) -> list[str]:
    """Extract the individual chunk texts from retrieved_guidance_json."""
    guidance_records = parse_json_list(
        value,
        "retrieved_guidance_json",
    )

    contexts = []

    for record in guidance_records:
        if not isinstance(record, dict):
            raise ValueError(
                "Each retrieved guidance record must be a dictionary."
            )

        chunk_text = str(record.get("chunk_text", "")).strip()

        if not chunk_text:
            raise ValueError(
                "A retrieved guidance record is missing chunk_text."
            )

        contexts.append(chunk_text)

    return contexts


def build_reference_answer(row: pd.Series) -> str:
    """Build an authoritative reference from deterministic facts only."""
    return (
        f"Claim {row['claim_id']} is routed to "
        f"{row['deterministic_outcome']} under deterministic rule "
        f"{row['triggering_rule_id']} at precedence stage "
        f"{int(row['precedence_stage'])}. "
        f"Reason: {str(row['deterministic_reason']).strip()} "
        "This is a triage recommendation for analyst decision support "
        "and is not an approval, payout, fraud determination, or final "
        "denial decision."
    )


ragas_input = generation_cases[
    [
        "evaluation_case_id",
        "claim_id",
        "deterministic_outcome",
        "triggering_rule_id",
        "precedence_stage",
        "controlled_query_text",
        "retrieved_guidance_json",
        "final_explanation",
    ]
].copy()

ragas_input["user_input"] = (
    ragas_input["controlled_query_text"]
    .astype(str)
    .str.strip()
)

ragas_input["retrieved_contexts"] = (
    ragas_input["retrieved_guidance_json"]
    .apply(extract_retrieved_contexts)
)

ragas_input["response"] = (
    ragas_input["final_explanation"]
    .astype(str)
    .str.strip()
)

ragas_input["reference"] = generation_cases.apply(
    build_reference_answer,
    axis=1,
)

assert ragas_input["user_input"].str.len().gt(0).all()
assert ragas_input["response"].str.len().gt(0).all()
assert ragas_input["reference"].str.len().gt(0).all()

assert ragas_input["retrieved_contexts"].apply(
    len
).eq(3).all()

assert ragas_input["retrieved_contexts"].apply(
    lambda contexts: all(
        isinstance(context, str) and context.strip()
        for context in contexts
    )
).all()

print("PASS: 12 Ragas input records created")
print("PASS: every record has a user input")
print("PASS: every record has three separate retrieved contexts")
print("PASS: every record has a frozen response")
print("PASS: every record has a deterministic reference answer")

display(
    ragas_input[
        [
            "evaluation_case_id",
            "claim_id",
            "deterministic_outcome",
            "triggering_rule_id",
            "user_input",
            "retrieved_contexts",
            "response",
            "reference",
        ]
    ].head(2)
)

PASS: 12 Ragas input records created
PASS: every record has a user input
PASS: every record has three separate retrieved contexts
PASS: every record has a frozen response
PASS: every record has a deterministic reference answer


,evaluation_case_id,claim_id,deterministic_outcome,triggering_rule_id,user_input,retrieved_contexts,response,reference
0,GEN-001,CLM-000001,PROCEED,OUT-001,Device protection claim analyst guidance. Auth...,[Document: Evidence Review Guide\nSection: Evi...,Claim CLM-000001 routed to PROCEED at preceden...,Claim CLM-000001 is routed to PROCEED under de...
1,GEN-002,CLM-000053,PROCEED,OUT-001,Device protection claim analyst guidance. Auth...,[Document: Evidence Review Guide\nSection: Evi...,Claim CLM-000053 routed to PROCEED at preceden...,Claim CLM-000053 is routed to PROCEED under de...


## 5. Validate Reference-Answer Suitability

The deterministic reference preserves claim-routing authority, but it is not suitable by itself for retrieval completeness metrics because claim-specific facts originate from structured tools rather than the retrieved knowledge-base chunks.

This section inspects the committed fields available for constructing a separate authoritative RAG-guidance reference.

In [7]:
# Inspect candidate authoritative fields without displaying all 66 columns.
candidate_terms = [
    "reason",
    "outcome",
    "rule",
    "precedence",
    "evidence",
    "follow",
    "guidance",
    "manual",
    "required",
    "expected",
    "reference",
    "retriev",
]

candidate_columns = [
    column
    for column in generation_cases.columns
    if any(term in column.lower() for term in candidate_terms)
]

print("Candidate columns:", len(candidate_columns))

for column in candidate_columns:
    print(column)

Candidate columns: 29
deterministic_outcome
triggering_rule_id
precedence_stage
deterministic_reason
rule_trace
projected_triage_outcome
projected_triggering_rule_id
projected_precedence_stage
evidence_profile_id
evidence_assessment_status
missing_required_evidence_codes
unreadable_required_evidence_codes
manual_review_reason_codes
follow_up_required
follow_up_selection_status
follow_up_question_ids
retrieval_status
retrieved_guidance_count
retrieved_chunk_ids
retrieved_document_ids
retrieved_context
retrieved_guidance_json
analyst_guidance_summary
analyst_guidance_usage_boundary
analyst_guidance_text
source_references_json
analyst_guidance_items_json
final_outcome
final_triggering_rule_id


In [8]:
# Inspect the structure of retrieved guidance for two representative cases.
for case_id in ["GEN-001", "GEN-005"]:
    row = generation_cases.loc[
        generation_cases["evaluation_case_id"].eq(case_id)
    ].iloc[0]

    guidance_records = parse_json_list(
        row["retrieved_guidance_json"],
        "retrieved_guidance_json",
    )

    print("=" * 100)
    print("Case:", case_id)
    print("Outcome:", row["deterministic_outcome"])
    print("Rule:", row["triggering_rule_id"])
    print("Deterministic reason:", row["deterministic_reason"])

    for index, record in enumerate(guidance_records, start=1):
        print(f"\nRetrieved context {index}")
        print("Chunk ID:", record.get("chunk_id"))
        print("Document:", record.get("document_title"))
        print("Section:", record.get("section_title"))
        print("Text:")
        print(record.get("chunk_text"))

Case: GEN-001
Outcome: PROCEED
Rule: OUT-001
Deterministic reason: All applicable deterministic checks passed. Eligible for standard processing only; this is not an approval, payout, fraud determination, or final denial decision.

Retrieved context 1
Chunk ID: KB-002::S03
Document: Evidence Review Guide
Section: Evidence profiles
Text:
Document: Evidence Review Guide
Section: Evidence profiles

Use the evidence profile linked to the applicable claim category:

| Claim category | Evidence profile | Primary required evidence |
|---|---|---|
| SCREEN_DAMAGE | EVD-SCREEN-01 | Damage photo |
| ACCIDENTAL_DAMAGE | EVD-ACCIDENTAL-01 | Damage photo |
| LIQUID_DAMAGE | EVD-LIQUID-01 | Damage photo |
| MECHANICAL_BREAKDOWN | EVD-MECHANICAL-01 | Diagnostic report |
| THEFT | EVD-THEFT-01 | Police report reference |

Repair quotes are supporting evidence where available and may be useful for repair feasibility or high-cost review. They are not automatically a reason to proceed or decline.

Retriev

## 6. Inspect Case-Level Guidance and Source Traceability

Review the deterministic routing reason, retrieved chunk IDs, formatted analyst guidance, and source references for all 12 cases.

This supports construction of a concise authoritative RAG-guidance reference without copying the generated final explanation.

In [9]:
def truncate_text(value: object, max_length: int = 220) -> str:
    """Return compact notebook display text without modifying source data."""
    text = "" if pd.isna(value) else str(value).strip()

    if len(text) <= max_length:
        return text

    return text[: max_length - 3] + "..."


case_guidance_inspection = generation_cases[
    [
        "evaluation_case_id",
        "claim_id",
        "deterministic_outcome",
        "triggering_rule_id",
        "precedence_stage",
        "deterministic_reason",
        "retrieved_chunk_ids",
        "analyst_guidance_summary",
        "analyst_guidance_usage_boundary",
        "analyst_guidance_text",
        "source_references_json",
    ]
].copy()

for column in [
    "deterministic_reason",
    "analyst_guidance_summary",
    "analyst_guidance_usage_boundary",
    "analyst_guidance_text",
]:
    case_guidance_inspection[column] = (
        case_guidance_inspection[column].apply(truncate_text)
    )

display(
    case_guidance_inspection.sort_values(
        "evaluation_case_id"
    ).reset_index(drop=True)
)

,evaluation_case_id,claim_id,deterministic_outcome,triggering_rule_id,precedence_stage,deterministic_reason,retrieved_chunk_ids,analyst_guidance_summary,analyst_guidance_usage_boundary,analyst_guidance_text,source_references_json
0,GEN-001,CLM-000001,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,"[""KB-002::S03"", ""KB-001::S01"", ""KB-002::S01""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Evide...,"[{""chunk_id"": ""KB-002::S03"", ""document_id"": ""K..."
1,GEN-002,CLM-000053,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,"[""KB-002::S03"", ""KB-001::S01"", ""KB-002::S01""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Evide...,"[{""chunk_id"": ""KB-002::S03"", ""document_id"": ""K..."
2,GEN-003,CLM-000177,PROCEED,OUT-001,6,All applicable deterministic checks passed. El...,"[""KB-002::S03"", ""KB-001::S01"", ""KB-002::S01""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Evide...,"[{""chunk_id"": ""KB-002::S03"", ""document_id"": ""K..."
3,GEN-004,CLM-000103,INFO_REQUIRED,DEV-001,3,"A usable device identifier, such as IMEI or se...","[""KB-002::S03"", ""KB-002::S02"", ""KB-002::S01""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Evide...,"[{""chunk_id"": ""KB-002::S03"", ""document_id"": ""K..."
4,GEN-005,CLM-000104,INFO_REQUIRED,DEV-001,3,"A usable device identifier, such as IMEI or se...","[""KB-002::S02"", ""KB-002::S03"", ""KB-002::S01""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Evide...,"[{""chunk_id"": ""KB-002::S02"", ""document_id"": ""K..."
5,GEN-006,CLM-000122,INFO_REQUIRED,EVD-001,5,Required evidence is missing or unreadable. Re...,"[""KB-002::S01"", ""KB-002::S03"", ""KB-002::S02""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Docum...,"[{""chunk_id"": ""KB-002::S01"", ""document_id"": ""K..."
6,GEN-007,CLM-000149,MANUAL_REVIEW,EVD-002,1,Submitted evidence contains a contradiction an...,"[""KB-002::S03"", ""KB-002::S02"", ""KB-002::S01""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Evide...,"[{""chunk_id"": ""KB-002::S03"", ""document_id"": ""K..."
7,GEN-008,CLM-000164,MANUAL_REVIEW,DATA-002,1,Multiple candidate policy records were found. ...,"[""KB-002::S01"", ""KB-005::S04"", ""KB-002::S03""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Docum...,"[{""chunk_id"": ""KB-002::S01"", ""document_id"": ""K..."
8,GEN-009,CLM-000180,MANUAL_REVIEW,ANM-001,4,A material risk or anomaly signal requires ana...,"[""KB-002::S01"", ""KB-002::S03"", ""KB-002::S02""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Evidence Review Guide Section: Docum...,"[{""chunk_id"": ""KB-002::S01"", ""document_id"": ""K..."
9,GEN-010,CLM-000154,NOT_ELIGIBLE,COV-001,2,The identified claim category is explicitly no...,"[""KB-005::S04"", ""KB-002::S02"", ""KB-002::S01""]",Retrieved 3 approved KB section(s) for analyst...,This analyst guidance is separate from final_r...,Document: Decision Explanation and Audit Guide...,"[{""chunk_id"": ""KB-005::S04"", ""document_id"": ""K..."


In [10]:
# Produce a compact rule-level view to determine whether references
# can be reused across cases sharing the same deterministic rule.
rule_summary = (
    generation_cases.groupby(
        [
            "triggering_rule_id",
            "deterministic_outcome",
            "precedence_stage",
            "deterministic_reason",
        ],
        dropna=False,
    )
    .agg(
        case_count=("evaluation_case_id", "count"),
        case_ids=(
            "evaluation_case_id",
            lambda values: sorted(values.tolist()),
        ),
        retrieved_chunk_sets=(
            "retrieved_chunk_ids",
            lambda values: sorted(set(values.astype(str))),
        ),
    )
    .reset_index()
    .sort_values(
        [
            "precedence_stage",
            "triggering_rule_id",
        ]
    )
)

print(
    "Unique deterministic rules represented:",
    rule_summary["triggering_rule_id"].nunique(),
)

display(rule_summary)

Unique deterministic rules represented: 8


,triggering_rule_id,deterministic_outcome,precedence_stage,deterministic_reason,case_count,case_ids,retrieved_chunk_sets
2,DATA-002,MANUAL_REVIEW,1,Multiple candidate policy records were found. ...,1,[GEN-008],"[[""KB-002::S01"", ""KB-005::S04"", ""KB-002::S03""]]"
5,EVD-002,MANUAL_REVIEW,1,Submitted evidence contains a contradiction an...,1,[GEN-007],"[[""KB-002::S03"", ""KB-002::S02"", ""KB-002::S01""]]"
1,COV-001,NOT_ELIGIBLE,2,The identified claim category is explicitly no...,2,"[GEN-010, GEN-011]","[[""KB-005::S04"", ""KB-002::S02"", ""KB-002::S01""]..."
6,LIM-001,NOT_ELIGIBLE,2,The plan's annual claim allowance is exhausted.,1,[GEN-012],"[[""KB-002::S02"", ""KB-002::S03"", ""KB-002::S01""]]"
3,DEV-001,INFO_REQUIRED,3,"A usable device identifier, such as IMEI or se...",2,"[GEN-004, GEN-005]","[[""KB-002::S02"", ""KB-002::S03"", ""KB-002::S01""]..."
0,ANM-001,MANUAL_REVIEW,4,A material risk or anomaly signal requires ana...,1,[GEN-009],"[[""KB-002::S01"", ""KB-002::S03"", ""KB-002::S02""]]"
4,EVD-001,INFO_REQUIRED,5,Required evidence is missing or unreadable. Re...,1,[GEN-006],"[[""KB-002::S01"", ""KB-002::S03"", ""KB-002::S02""]]"
7,OUT-001,PROCEED,6,All applicable deterministic checks passed. El...,3,"[GEN-001, GEN-002, GEN-003]","[[""KB-002::S03"", ""KB-001::S01"", ""KB-002::S01""]]"


In [11]:
# Validate and inspect the source-reference structure.
def parse_json_object_or_list(
    value: object,
    field_name: str,
) -> object:
    if pd.isna(value):
        return None

    try:
        return json.loads(str(value))
    except json.JSONDecodeError as error:
        raise ValueError(
            f"{field_name} contains invalid JSON."
        ) from error


source_reference_records = []

for row in generation_cases.itertuples(index=False):
    parsed_references = parse_json_object_or_list(
        row.source_references_json,
        "source_references_json",
    )

    source_reference_records.append(
        {
            "evaluation_case_id": row.evaluation_case_id,
            "triggering_rule_id": row.triggering_rule_id,
            "retrieved_chunk_ids": row.retrieved_chunk_ids,
            "source_references": parsed_references,
        }
    )

source_reference_inspection = pd.DataFrame(
    source_reference_records
)

display(source_reference_inspection)

print("PASS: source references parsed for all 12 cases")

,evaluation_case_id,triggering_rule_id,retrieved_chunk_ids,source_references
0,GEN-001,OUT-001,"[""KB-002::S03"", ""KB-001::S01"", ""KB-002::S01""]","[{'chunk_id': 'KB-002::S03', 'document_id': 'K..."
1,GEN-002,OUT-001,"[""KB-002::S03"", ""KB-001::S01"", ""KB-002::S01""]","[{'chunk_id': 'KB-002::S03', 'document_id': 'K..."
2,GEN-003,OUT-001,"[""KB-002::S03"", ""KB-001::S01"", ""KB-002::S01""]","[{'chunk_id': 'KB-002::S03', 'document_id': 'K..."
3,GEN-004,DEV-001,"[""KB-002::S03"", ""KB-002::S02"", ""KB-002::S01""]","[{'chunk_id': 'KB-002::S03', 'document_id': 'K..."
4,GEN-005,DEV-001,"[""KB-002::S02"", ""KB-002::S03"", ""KB-002::S01""]","[{'chunk_id': 'KB-002::S02', 'document_id': 'K..."
5,GEN-006,EVD-001,"[""KB-002::S01"", ""KB-002::S03"", ""KB-002::S02""]","[{'chunk_id': 'KB-002::S01', 'document_id': 'K..."
6,GEN-007,EVD-002,"[""KB-002::S03"", ""KB-002::S02"", ""KB-002::S01""]","[{'chunk_id': 'KB-002::S03', 'document_id': 'K..."
7,GEN-008,DATA-002,"[""KB-002::S01"", ""KB-005::S04"", ""KB-002::S03""]","[{'chunk_id': 'KB-002::S01', 'document_id': 'K..."
8,GEN-009,ANM-001,"[""KB-002::S01"", ""KB-002::S03"", ""KB-002::S02""]","[{'chunk_id': 'KB-002::S01', 'document_id': 'K..."
9,GEN-010,COV-001,"[""KB-005::S04"", ""KB-002::S02"", ""KB-002::S01""]","[{'chunk_id': 'KB-005::S04', 'document_id': 'K..."


PASS: source references parsed for all 12 cases


## 7. Freeze Rule-Level RAG Guidance References

Each deterministic rule is mapped to an authoritative operational-guidance reference.

These references describe what useful KB retrieval should supply to the generated analyst explanation. They:

- exclude claim-specific facts obtained from structured tools,
- do not copy the frozen generated response,
- do not change deterministic outcomes,
- are grounded in the approved operational knowledge base,
- are frozen before running Ragas metrics.

In [12]:
RAG_GUIDANCE_REFERENCE_BY_RULE = {
    "OUT-001": (
        "A claim may proceed to standard processing only when all applicable "
        "deterministic controls have passed. The explanation should make clear "
        "that this is a triage recommendation, not final approval, payment "
        "authorization, or a binding customer commitment."
    ),
    "DEV-001": (
        "A usable device identifier such as an IMEI or serial number must not "
        "be inferred from vague or incomplete information. Request only the "
        "missing identifier needed for the next decision step, and state that "
        "additional information is required before assessment can continue."
    ),
    "EVD-001": (
        "Missing mandatory evidence requires a targeted request for the "
        "specific required document. Unreadable evidence requires a clearer "
        "or complete re-upload. Evidence validates a potentially eligible "
        "claim but does not create coverage."
    ),
    "EVD-002": (
        "Contradictory or materially invalid evidence requires analyst review. "
        "The inconsistency should be described neutrally using verified facts, "
        "without inferring fraud, dishonesty, or issuing an automated decline."
    ),
    "DATA-002": (
        "When multiple or conflicting authoritative records are found, the "
        "system must not choose one based on narrative plausibility. The case "
        "should follow the configured data-conflict manual-review path, with "
        "the unresolved records and source references clearly identified."
    ),
    "ANM-001": (
        "A configured material anomaly or risk trigger requires analyst review. "
        "The escalation should contain verified facts, the review reason, rule "
        "and source references, and unresolved questions. The agent must not "
        "make a fraud determination or conduct the analyst investigation."
    ),
    "COV-001": (
        "When configured policy rules establish that the reported category is "
        "not covered, the result should be described as a recommendation for "
        "decline confirmation based on verified information and configured "
        "rules. Evidence must not be used to create coverage, and the agent "
        "must not state a final denial."
    ),
    "LIM-001": (
        "Claim-limit consumption is assessed using authoritative prior-claims "
        "history and configured policy rules. When the applicable allowance is "
        "exhausted, the result should be explained using the verified limit "
        "facts and rule references, without presenting it as a final denial."
    ),
}

EXPECTED_RULE_IDS = set(
    generation_cases["triggering_rule_id"].unique()
)

assert set(RAG_GUIDANCE_REFERENCE_BY_RULE) == EXPECTED_RULE_IDS

assert all(
    isinstance(reference, str) and reference.strip()
    for reference in RAG_GUIDANCE_REFERENCE_BY_RULE.values()
)

print(
    "PASS: guidance references defined for",
    len(RAG_GUIDANCE_REFERENCE_BY_RULE),
    "deterministic rules",
)

for rule_id, reference in RAG_GUIDANCE_REFERENCE_BY_RULE.items():
    print(f"\n{rule_id}")
    print(reference)

PASS: guidance references defined for 8 deterministic rules

OUT-001
A claim may proceed to standard processing only when all applicable deterministic controls have passed. The explanation should make clear that this is a triage recommendation, not final approval, payment authorization, or a binding customer commitment.

DEV-001
A usable device identifier such as an IMEI or serial number must not be inferred from vague or incomplete information. Request only the missing identifier needed for the next decision step, and state that additional information is required before assessment can continue.

EVD-001
Missing mandatory evidence requires a targeted request for the specific required document. Unreadable evidence requires a clearer or complete re-upload. Evidence validates a potentially eligible claim but does not create coverage.

EVD-002
Contradictory or materially invalid evidence requires analyst review. The inconsistency should be described neutrally using verified facts, without 

In [13]:
# Map the frozen rule references to all 12 evaluation cases.
ragas_input["deterministic_reference"] = ragas_input["reference"]

ragas_input["rag_guidance_reference"] = (
    ragas_input["triggering_rule_id"].map(
        RAG_GUIDANCE_REFERENCE_BY_RULE
    )
)

assert ragas_input["rag_guidance_reference"].notna().all()
assert ragas_input["rag_guidance_reference"].str.len().gt(0).all()

# Ragas uses the RAG guidance reference for retrieval-quality metrics.
ragas_input["reference"] = ragas_input[
    "rag_guidance_reference"
]

assert not ragas_input["reference"].eq(
    ragas_input["response"]
).any()

assert not ragas_input["deterministic_reference"].eq(
    ragas_input["rag_guidance_reference"]
).any()

print("PASS: all 12 cases mapped to a rule-level RAG reference")
print("PASS: deterministic and RAG references remain separate")
print("PASS: no reference is copied from the generated response")

display(
    ragas_input[
        [
            "evaluation_case_id",
            "triggering_rule_id",
            "deterministic_reference",
            "rag_guidance_reference",
        ]
    ]
)

PASS: all 12 cases mapped to a rule-level RAG reference
PASS: deterministic and RAG references remain separate
PASS: no reference is copied from the generated response


,evaluation_case_id,triggering_rule_id,deterministic_reference,rag_guidance_reference
0,GEN-001,OUT-001,Claim CLM-000001 is routed to PROCEED under de...,A claim may proceed to standard processing onl...
1,GEN-002,OUT-001,Claim CLM-000053 is routed to PROCEED under de...,A claim may proceed to standard processing onl...
2,GEN-003,OUT-001,Claim CLM-000177 is routed to PROCEED under de...,A claim may proceed to standard processing onl...
3,GEN-004,DEV-001,Claim CLM-000103 is routed to INFO_REQUIRED un...,A usable device identifier such as an IMEI or ...
4,GEN-005,DEV-001,Claim CLM-000104 is routed to INFO_REQUIRED un...,A usable device identifier such as an IMEI or ...
5,GEN-006,EVD-001,Claim CLM-000122 is routed to INFO_REQUIRED un...,Missing mandatory evidence requires a targeted...
6,GEN-007,EVD-002,Claim CLM-000149 is routed to MANUAL_REVIEW un...,Contradictory or materially invalid evidence r...
7,GEN-008,DATA-002,Claim CLM-000164 is routed to MANUAL_REVIEW un...,When multiple or conflicting authoritative rec...
8,GEN-009,ANM-001,Claim CLM-000180 is routed to MANUAL_REVIEW un...,A configured material anomaly or risk trigger ...
9,GEN-010,COV-001,Claim CLM-000154 is routed to NOT_ELIGIBLE und...,When configured policy rules establish that th...


## 8. Validate Reference Provenance and Coverage

The rule-level references are manually authored evaluation labels derived from the approved knowledge base.

They are used only for automated evaluation and do not enter the production workflow, retrieval query, generated response, or deterministic routing logic.

In [14]:
REFERENCE_PROVENANCE_BY_RULE = {
    "OUT-001": [
        "KB-001::S04",
        "KB-001::S06",
        "KB-005::S03",
    ],
    "DEV-001": [
        "KB-001::S05",
        "KB-005::S03",
    ],
    "EVD-001": [
        "KB-002::S02",
        "KB-002::S04",
        "KB-002::S05",
    ],
    "EVD-002": [
        "KB-002::S02",
        "KB-002::S04",
        "KB-002::S06",
        "KB-004::S03",
    ],
    "DATA-002": [
        "KB-001::S03",
        "KB-004::S01",
        "KB-004::S02",
    ],
    "ANM-001": [
        "KB-004::S01",
        "KB-004::S02",
        "KB-004::S03",
        "KB-004::S04",
    ],
    "COV-001": [
        "KB-002::S02",
        "KB-005::S03",
    ],
    "LIM-001": [
        "KB-001::S03",
        "KB-001::S04",
        "KB-005::S03",
    ],
}

assert set(REFERENCE_PROVENANCE_BY_RULE) == EXPECTED_RULE_IDS

reference_provenance = pd.DataFrame(
    [
        {
            "triggering_rule_id": rule_id,
            "rag_guidance_reference": (
                RAG_GUIDANCE_REFERENCE_BY_RULE[rule_id]
            ),
            "reference_source_chunk_ids": source_chunk_ids,
        }
        for rule_id, source_chunk_ids
        in REFERENCE_PROVENANCE_BY_RULE.items()
    ]
)

assert reference_provenance["reference_source_chunk_ids"].apply(
    len
).ge(1).all()

print("PASS: provenance recorded for all 8 rule references")

display(reference_provenance)

PASS: provenance recorded for all 8 rule references


,triggering_rule_id,rag_guidance_reference,reference_source_chunk_ids
0,OUT-001,A claim may proceed to standard processing onl...,"[KB-001::S04, KB-001::S06, KB-005::S03]"
1,DEV-001,A usable device identifier such as an IMEI or ...,"[KB-001::S05, KB-005::S03]"
2,EVD-001,Missing mandatory evidence requires a targeted...,"[KB-002::S02, KB-002::S04, KB-002::S05]"
3,EVD-002,Contradictory or materially invalid evidence r...,"[KB-002::S02, KB-002::S04, KB-002::S06, KB-004..."
4,DATA-002,When multiple or conflicting authoritative rec...,"[KB-001::S03, KB-004::S01, KB-004::S02]"
5,ANM-001,A configured material anomaly or risk trigger ...,"[KB-004::S01, KB-004::S02, KB-004::S03, KB-004..."
6,COV-001,When configured policy rules establish that th...,"[KB-002::S02, KB-005::S03]"
7,LIM-001,Claim-limit consumption is assessed using auth...,"[KB-001::S03, KB-001::S04, KB-005::S03]"


In [15]:
def parse_chunk_id_list(value: object) -> list[str]:
    """Parse the committed retrieved_chunk_ids JSON list."""
    parsed_value = json.loads(str(value))

    if not isinstance(parsed_value, list):
        raise ValueError(
            "retrieved_chunk_ids must contain a JSON list."
        )

    return [str(item) for item in parsed_value]


ragas_input["retrieved_chunk_ids_list"] = (
    generation_cases["retrieved_chunk_ids"].apply(
        parse_chunk_id_list
    )
)

ragas_input["reference_source_chunk_ids"] = (
    ragas_input["triggering_rule_id"].map(
        REFERENCE_PROVENANCE_BY_RULE
    )
)


def calculate_reference_chunk_overlap(
    retrieved_ids: list[str],
    reference_ids: list[str],
) -> list[str]:
    return sorted(
        set(retrieved_ids).intersection(reference_ids)
    )


ragas_input["reference_chunks_retrieved"] = (
    ragas_input.apply(
        lambda row: calculate_reference_chunk_overlap(
            row["retrieved_chunk_ids_list"],
            row["reference_source_chunk_ids"],
        ),
        axis=1,
    )
)

ragas_input["reference_chunk_hit"] = (
    ragas_input["reference_chunks_retrieved"]
    .apply(bool)
)

reference_hit_summary = (
    ragas_input.groupby(
        [
            "triggering_rule_id",
            "deterministic_outcome",
        ],
        dropna=False,
    )
    .agg(
        case_count=("evaluation_case_id", "count"),
        reference_chunk_hits=(
            "reference_chunk_hit",
            "sum",
        ),
        case_ids=(
            "evaluation_case_id",
            lambda values: sorted(values.tolist()),
        ),
    )
    .reset_index()
)

reference_hit_summary["reference_chunk_hit_rate"] = (
    reference_hit_summary["reference_chunk_hits"]
    / reference_hit_summary["case_count"]
)

display(
    ragas_input[
        [
            "evaluation_case_id",
            "triggering_rule_id",
            "retrieved_chunk_ids_list",
            "reference_source_chunk_ids",
            "reference_chunks_retrieved",
            "reference_chunk_hit",
        ]
    ]
)

display(reference_hit_summary)

print(
    "Cases with at least one ideal reference chunk retrieved:",
    int(ragas_input["reference_chunk_hit"].sum()),
    "of",
    EXPECTED_CASE_COUNT,
)

,evaluation_case_id,triggering_rule_id,retrieved_chunk_ids_list,reference_source_chunk_ids,reference_chunks_retrieved,reference_chunk_hit
0,GEN-001,OUT-001,"[KB-002::S03, KB-001::S01, KB-002::S01]","[KB-001::S04, KB-001::S06, KB-005::S03]",[],False
1,GEN-002,OUT-001,"[KB-002::S03, KB-001::S01, KB-002::S01]","[KB-001::S04, KB-001::S06, KB-005::S03]",[],False
2,GEN-003,OUT-001,"[KB-002::S03, KB-001::S01, KB-002::S01]","[KB-001::S04, KB-001::S06, KB-005::S03]",[],False
3,GEN-004,DEV-001,"[KB-002::S03, KB-002::S02, KB-002::S01]","[KB-001::S05, KB-005::S03]",[],False
4,GEN-005,DEV-001,"[KB-002::S02, KB-002::S03, KB-002::S01]","[KB-001::S05, KB-005::S03]",[],False
5,GEN-006,EVD-001,"[KB-002::S01, KB-002::S03, KB-002::S02]","[KB-002::S02, KB-002::S04, KB-002::S05]",[KB-002::S02],True
6,GEN-007,EVD-002,"[KB-002::S03, KB-002::S02, KB-002::S01]","[KB-002::S02, KB-002::S04, KB-002::S06, KB-004...",[KB-002::S02],True
7,GEN-008,DATA-002,"[KB-002::S01, KB-005::S04, KB-002::S03]","[KB-001::S03, KB-004::S01, KB-004::S02]",[],False
8,GEN-009,ANM-001,"[KB-002::S01, KB-002::S03, KB-002::S02]","[KB-004::S01, KB-004::S02, KB-004::S03, KB-004...",[],False
9,GEN-010,COV-001,"[KB-005::S04, KB-002::S02, KB-002::S01]","[KB-002::S02, KB-005::S03]",[KB-002::S02],True


,triggering_rule_id,deterministic_outcome,case_count,reference_chunk_hits,case_ids,reference_chunk_hit_rate
0,ANM-001,MANUAL_REVIEW,1,0,[GEN-009],0.0
1,COV-001,NOT_ELIGIBLE,2,1,"[GEN-010, GEN-011]",0.5
2,DATA-002,MANUAL_REVIEW,1,0,[GEN-008],0.0
3,DEV-001,INFO_REQUIRED,2,0,"[GEN-004, GEN-005]",0.0
4,EVD-001,INFO_REQUIRED,1,1,[GEN-006],1.0
5,EVD-002,MANUAL_REVIEW,1,1,[GEN-007],1.0
6,LIM-001,NOT_ELIGIBLE,1,0,[GEN-012],0.0
7,OUT-001,PROCEED,3,0,"[GEN-001, GEN-002, GEN-003]",0.0


Cases with at least one ideal reference chunk retrieved: 3 of 12


## 9. Validate Ragas Metric APIs

Confirm the metric classes available in the pinned Ragas 0.3.9 environment before configuring evaluation.

No model or API call is performed in this section.

In [16]:
from datasets import Dataset
from ragas import EvaluationDataset, SingleTurnSample, evaluate
import ragas.metrics as ragas_metrics


available_metric_names = sorted(
    name
    for name in dir(ragas_metrics)
    if any(
        term in name.lower()
        for term in [
            "faith",
            "relev",
            "precision",
            "recall",
        ]
    )
)

print("Relevant Ragas metric exports:")

for name in available_metric_names:
    print(name)

Relevant Ragas metric exports:
AnswerRelevancy
ContextEntityRecall
ContextPrecision
ContextRecall
ContextRelevance
Faithfulness
FaithfulnesswithHHEM
IDBasedContextPrecision
IDBasedContextRecall
LLMContextPrecisionWithReference
LLMContextPrecisionWithoutReference
LLMContextRecall
MultiModalFaithfulness
MultiModalRelevance
NonLLMContextPrecisionWithReference
NonLLMContextRecall
ResponseRelevancy
_answer_relevance
_context_entities_recall
_context_precision
_context_recall
_faithfulness
_multi_modal_faithfulness
_multi_modal_relevance
answer_relevancy
context_entity_recall
context_precision
context_recall
faithfulness
multimodal_faithness
multimodal_relevance


In [17]:
# Import the four agreed automated RAG metrics.
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)

selected_metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
]

print("Selected metrics:")

for metric in selected_metrics:
    print(
        "-",
        metric.name,
        "|",
        type(metric).__name__,
    )

assert len(selected_metrics) == 4
assert {metric.name for metric in selected_metrics} == {
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
}

print("PASS: four required Ragas metrics imported")

Selected metrics:
- faithfulness | Faithfulness
- answer_relevancy | AnswerRelevancy
- context_precision | ContextPrecision
- context_recall | ContextRecall
PASS: four required Ragas metrics imported


## 10. Build the Frozen Ragas Evaluation Dataset

Convert the validated 12-case adapter into Ragas single-turn samples.

The dataset uses the frozen workflow inputs and outputs and the manually reviewed rule-level RAG-guidance references.

In [18]:
ragas_samples = [
    SingleTurnSample(
        user_input=row.user_input,
        retrieved_contexts=row.retrieved_contexts,
        response=row.response,
        reference=row.reference,
    )
    for row in ragas_input.itertuples(index=False)
]

evaluation_dataset = EvaluationDataset(
    samples=ragas_samples
)

assert len(evaluation_dataset) == EXPECTED_CASE_COUNT

print(
    "PASS: Ragas evaluation dataset created with",
    len(evaluation_dataset),
    "samples",
)

first_sample = evaluation_dataset[0]

print("\nFirst sample fields:")
print("User input length:", len(first_sample.user_input))
print(
    "Retrieved contexts:",
    len(first_sample.retrieved_contexts),
)
print("Response length:", len(first_sample.response))
print("Reference length:", len(first_sample.reference))

PASS: Ragas evaluation dataset created with 12 samples

First sample fields:
User input length: 888
Retrieved contexts: 3
Response length: 885
Reference length: 252


In [19]:
# Verify round-trip conversion without exposing credentials or changing data.
evaluation_dataset_frame = evaluation_dataset.to_pandas()

print(
    "Evaluation dataset shape:",
    evaluation_dataset_frame.shape,
)

print(
    "Evaluation dataset columns:",
    evaluation_dataset_frame.columns.tolist(),
)

assert len(evaluation_dataset_frame) == EXPECTED_CASE_COUNT

assert {
    "user_input",
    "retrieved_contexts",
    "response",
    "reference",
}.issubset(
    evaluation_dataset_frame.columns
)

assert evaluation_dataset_frame[
    "retrieved_contexts"
].apply(len).eq(3).all()

print("PASS: evaluation dataset round-trip validated")

display(evaluation_dataset_frame.head(2))

Evaluation dataset shape: (12, 4)
Evaluation dataset columns: ['user_input', 'retrieved_contexts', 'response', 'reference']
PASS: evaluation dataset round-trip validated


,user_input,retrieved_contexts,response,reference
0,Device protection claim analyst guidance. Auth...,[Document: Evidence Review Guide\nSection: Evi...,Claim CLM-000001 routed to PROCEED at preceden...,A claim may proceed to standard processing onl...
1,Device protection claim analyst guidance. Auth...,[Document: Evidence Review Guide\nSection: Evi...,Claim CLM-000053 routed to PROCEED at preceden...,A claim may proceed to standard processing onl...


## 11. Configure Automated Evaluator Models

Ragas requires an evaluator LLM for claim and relevance judgements and an embedding model for answer-relevancy similarity.

The models are used only to score frozen outputs. They do not generate new workflow answers or alter claim decisions.

In [20]:
from dotenv import load_dotenv
from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings,
)


load_dotenv(PROJECT_ROOT / ".env")

assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY was not found in the environment or project .env file."
)

EVALUATOR_LLM_MODEL = "gpt-5.4-mini"
EVALUATOR_EMBEDDING_MODEL = "text-embedding-3-small"

evaluator_llm = ChatOpenAI(
    model=EVALUATOR_LLM_MODEL,
    temperature=0,
    max_retries=2,
    timeout=120,
)

evaluator_embeddings = OpenAIEmbeddings(
    model=EVALUATOR_EMBEDDING_MODEL,
    max_retries=2,
    request_timeout=120,
)

print("PASS: OpenAI API key is available")
print("Evaluator LLM:", EVALUATOR_LLM_MODEL)
print(
    "Evaluator embeddings:",
    EVALUATOR_EMBEDDING_MODEL,
)
print("PASS: evaluator clients configured")

PASS: OpenAI API key is available
Evaluator LLM: gpt-5.4-mini
Evaluator embeddings: text-embedding-3-small
PASS: evaluator clients configured


## 12. One-Case Ragas Smoke Test

Run all four metrics on one representative development case before evaluating the complete dataset.

This validates the metric, model, dataset, and API integration while limiting cost and making failures easier to diagnose.

In [21]:
SMOKE_TEST_CASE_ID = "GEN-006"

smoke_test_position = ragas_input.index[
    ragas_input["evaluation_case_id"].eq(
        SMOKE_TEST_CASE_ID
    )
].tolist()

assert len(smoke_test_position) == 1

smoke_test_index = smoke_test_position[0]

smoke_test_dataset = EvaluationDataset(
    samples=[
        ragas_samples[smoke_test_index]
    ]
)

print("Smoke-test case:", SMOKE_TEST_CASE_ID)
print(
    "Rule:",
    ragas_input.loc[
        smoke_test_index,
        "triggering_rule_id",
    ],
)
print(
    "Reference chunk hit:",
    ragas_input.loc[
        smoke_test_index,
        "reference_chunk_hit",
    ],
)
print(
    "Retrieved chunks:",
    ragas_input.loc[
        smoke_test_index,
        "retrieved_chunk_ids_list",
    ],
)

Smoke-test case: GEN-006
Rule: EVD-001
Reference chunk hit: True
Retrieved chunks: ['KB-002::S01', 'KB-002::S03', 'KB-002::S02']


In [22]:
from ragas.run_config import RunConfig


RAGAS_RUN_CONFIG = RunConfig(
    timeout=120,
    max_retries=2,
    max_wait=60,
    max_workers=1,
)

print("PASS: Ragas run configuration created")
print("Timeout:", RAGAS_RUN_CONFIG.timeout)
print("Max retries:", RAGAS_RUN_CONFIG.max_retries)
print("Max workers:", RAGAS_RUN_CONFIG.max_workers)

PASS: Ragas run configuration created
Timeout: 120
Max retries: 2
Max workers: 1


## 13. Execute One-Case Automated Evaluation

Run the four selected Ragas metrics on the representative `GEN-006` case.

This is the first billable evaluation call in Notebook 09. It scores an already frozen response and does not regenerate the workflow output.

In [23]:
smoke_test_result = evaluate(
    dataset=smoke_test_dataset,
    metrics=selected_metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RAGAS_RUN_CONFIG,
    raise_exceptions=False,
    show_progress=True,
)

print("PASS: one-case Ragas smoke test completed")
print(smoke_test_result)

Evaluating: 100%|██████████| 4/4 [00:34<00:00,  8.55s/it]

PASS: one-case Ragas smoke test completed
{'faithfulness': 0.2222, 'answer_relevancy': 0.5705, 'context_precision': 0.5833, 'context_recall': 1.0000}


In [24]:
smoke_test_result_frame = smoke_test_result.to_pandas()

print("Smoke-test result shape:", smoke_test_result_frame.shape)
print(
    "Smoke-test result columns:",
    smoke_test_result_frame.columns.tolist(),
)

expected_metric_columns = {
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
}

missing_metric_columns = (
    expected_metric_columns
    - set(smoke_test_result_frame.columns)
)

assert not missing_metric_columns, (
    "Smoke-test output missing metric columns: "
    + ", ".join(sorted(missing_metric_columns))
)

metric_values = smoke_test_result_frame[
    sorted(expected_metric_columns)
].iloc[0]

assert metric_values.notna().all(), (
    "One or more Ragas metrics returned NaN. "
    "Review the warnings or execution logs before continuing."
)

assert metric_values.between(0.0, 1.0).all(), (
    "Ragas metric scores must be between 0 and 1."
)

print("PASS: all four smoke-test metric scores are present")
print("PASS: all metric scores are within the expected 0–1 range")

display(
    smoke_test_result_frame[
        [
            "faithfulness",
            "answer_relevancy",
            "context_precision",
            "context_recall",
        ]
    ]
)

Smoke-test result shape: (1, 8)
Smoke-test result columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
PASS: all four smoke-test metric scores are present
PASS: all metric scores are within the expected 0–1 range


,faithfulness,answer_relevancy,context_precision,context_recall
0,0.222222,0.57054,0.583333,1.0


## 14. Adapt Faithfulness Evaluation to the Hybrid Architecture

The initial smoke test evaluated the full generated explanation against retrieved KB context alone.

However, the workflow generates explanations from two authoritative inputs:

1. deterministic structured facts produced by validated tools, and
2. retrieved knowledge-base guidance.

Claim-specific statements such as the claim ID, deterministic outcome, triggering rule, precedence stage, and deterministic reason are not expected to appear in the KB chunks. Therefore, evaluating the complete response against KB context alone underestimates faithfulness.

The following evaluation separates:

- **retrieval quality**, using only retrieved KB contexts, and
- **response faithfulness**, using the complete authoritative support context available during generation.

The frozen response is not modified or regenerated.

In [25]:
def build_structured_support_context(row: pd.Series) -> str:
    """
    Construct the structured deterministic facts that were available
    to the LLM before RAG guidance was added.
    """

    return f"""
Claim ID:
{row.claim_id}

Deterministic Outcome:
{row.deterministic_outcome}

Triggering Rule:
{row.triggering_rule_id}

Precedence Stage:
{row.precedence_stage}

Deterministic Reason:
{generation_cases.loc[
    generation_cases.evaluation_case_id == row.evaluation_case_id,
    "deterministic_reason"
].iloc[0]}
""".strip()


ragas_input["structured_support_context"] = (
    ragas_input.apply(
        build_structured_support_context,
        axis=1,
    )
)

ragas_input["supporting_contexts"] = (
    ragas_input.apply(
        lambda row: (
            [row["structured_support_context"]]
            + row["retrieved_contexts"]
        ),
        axis=1,
    )
)

assert ragas_input["supporting_contexts"].apply(len).eq(4).all()

print("PASS: hybrid supporting contexts created")

display(
    ragas_input[
        [
            "evaluation_case_id",
            "supporting_contexts",
        ]
    ].head(2)
)

PASS: hybrid supporting contexts created


,evaluation_case_id,supporting_contexts
0,GEN-001,[Claim ID:\nCLM-000001\n\nDeterministic Outcom...
1,GEN-002,[Claim ID:\nCLM-000053\n\nDeterministic Outcom...


In [26]:
# Build separate datasets for retrieval quality and response quality.

retrieval_quality_samples = [
    SingleTurnSample(
        user_input=row.user_input,
        retrieved_contexts=row.retrieved_contexts,
        response=row.response,
        reference=row.rag_guidance_reference,
    )
    for row in ragas_input.itertuples(index=False)
]

response_quality_samples = [
    SingleTurnSample(
        user_input=row.user_input,
        retrieved_contexts=row.supporting_contexts,
        response=row.response,
        reference=row.rag_guidance_reference,
    )
    for row in ragas_input.itertuples(index=False)
]

retrieval_quality_dataset = EvaluationDataset(
    samples=retrieval_quality_samples
)

response_quality_dataset = EvaluationDataset(
    samples=response_quality_samples
)

assert len(retrieval_quality_dataset) == EXPECTED_CASE_COUNT
assert len(response_quality_dataset) == EXPECTED_CASE_COUNT

assert all(
    len(sample.retrieved_contexts) == 3
    for sample in retrieval_quality_dataset
)

assert all(
    len(sample.retrieved_contexts) == 4
    for sample in response_quality_dataset
)

print("PASS: retrieval-quality dataset created")
print("PASS: response-quality dataset created")
print(
    "Retrieval contexts per case:",
    len(retrieval_quality_dataset[0].retrieved_contexts),
)
print(
    "Response-support contexts per case:",
    len(response_quality_dataset[0].retrieved_contexts),
)

PASS: retrieval-quality dataset created
PASS: response-quality dataset created
Retrieval contexts per case: 3
Response-support contexts per case: 4


## 15. Adjusted One-Case Response-Quality Smoke Test

Re-evaluate `GEN-006` using the complete authoritative support context:

- deterministic structured facts,
- three retrieved KB chunks.

Only Faithfulness and Answer Relevancy are evaluated in this pass.

In [27]:
adjusted_smoke_test_dataset = EvaluationDataset(
    samples=[
        response_quality_samples[smoke_test_index]
    ]
)

response_quality_metrics = [
    faithfulness,
    answer_relevancy,
]

adjusted_smoke_test_result = evaluate(
    dataset=adjusted_smoke_test_dataset,
    metrics=response_quality_metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RAGAS_RUN_CONFIG,
    raise_exceptions=False,
    show_progress=True,
)

print("PASS: adjusted response-quality smoke test completed")
print(adjusted_smoke_test_result)

Evaluating: 100%|██████████| 2/2 [00:09<00:00,  4.61s/it]

PASS: adjusted response-quality smoke test completed
{'faithfulness': 0.2000, 'answer_relevancy': 0.5458}


In [28]:
adjusted_smoke_test_frame = (
    adjusted_smoke_test_result.to_pandas()
)

required_response_metric_columns = {
    "faithfulness",
    "answer_relevancy",
}

missing_columns = (
    required_response_metric_columns
    - set(adjusted_smoke_test_frame.columns)
)

assert not missing_columns, (
    "Adjusted smoke test missing columns: "
    + ", ".join(sorted(missing_columns))
)

adjusted_metric_values = adjusted_smoke_test_frame[
    sorted(required_response_metric_columns)
].iloc[0]

assert adjusted_metric_values.notna().all()
assert adjusted_metric_values.between(0.0, 1.0).all()

original_faithfulness = float(
    smoke_test_result_frame.loc[0, "faithfulness"]
)

adjusted_faithfulness = float(
    adjusted_smoke_test_frame.loc[0, "faithfulness"]
)

print("PASS: adjusted metric scores are valid")
print(
    "Original KB-only faithfulness:",
    round(original_faithfulness, 6),
)
print(
    "Adjusted hybrid-context faithfulness:",
    round(adjusted_faithfulness, 6),
)
print(
    "Faithfulness change:",
    round(
        adjusted_faithfulness
        - original_faithfulness,
        6,
    ),
)

display(
    adjusted_smoke_test_frame[
        [
            "faithfulness",
            "answer_relevancy",
        ]
    ]
)

PASS: adjusted metric scores are valid
Original KB-only faithfulness: 0.222222
Adjusted hybrid-context faithfulness: 0.2
Faithfulness change: -0.022222


,faithfulness,answer_relevancy
0,0.2,0.545838


## 16. Diagnose the Adjusted Faithfulness Result

Adding a compact deterministic support context did not increase the `GEN-006`
Faithfulness score.

Before running the full evaluation, inspect the exact frozen response and the
authoritative structured facts available to generation. This determines whether
the low score reflects incomplete evaluator context, unsupported response
wording, or evaluator variability.

In [29]:
diagnostic_case_id = "GEN-006"

diagnostic_generation_row = generation_cases.loc[
    generation_cases["evaluation_case_id"].eq(diagnostic_case_id)
].iloc[0]

diagnostic_ragas_row = ragas_input.loc[
    ragas_input["evaluation_case_id"].eq(diagnostic_case_id)
].iloc[0]

print("=" * 100)
print("FROZEN RESPONSE")
print("=" * 100)
print(diagnostic_ragas_row["response"])

print("\n" + "=" * 100)
print("COMPACT STRUCTURED SUPPORT CONTEXT USED IN ADJUSTED TEST")
print("=" * 100)
print(diagnostic_ragas_row["structured_support_context"])

print("\n" + "=" * 100)
print("RETRIEVED KB CONTEXTS")
print("=" * 100)

for context_number, context in enumerate(
    diagnostic_ragas_row["retrieved_contexts"],
    start=1,
):
    print(f"\n--- Context {context_number} ---")
    print(context)

FROZEN RESPONSE
Claim CLM-000122 routed to INFO_REQUIRED at precedence stage 5. The trigger was EVD-001 because the evidence assessment is INCOMPLETE for profile EVD-MECHANICAL-01, with missing required evidence type DIAGNOSTIC_REPORT. Earlier rule checks were not triggered.

This is a system triage recommendation only. Authorised reviewers may apply approved review, escalation, or exception-handling procedures using the authoritative decision record. System limitations noted: DEV-003, EXC-001/EXC-002, and claim history completeness cannot be independently verified.

Request only the specific missing evidence identified in the rule trace: DIAGNOSTIC_REPORT. Then continue review under the standard authorised process.

COMPACT STRUCTURED SUPPORT CONTEXT USED IN ADJUSTED TEST
Claim ID:
CLM-000122

Deterministic Outcome:
INFO_REQUIRED

Triggering Rule:
EVD-001

Precedence Stage:
5

Deterministic Reason:
Required evidence is missing or unreadable. Request only the specific evidence items id

In [30]:
# Display all non-empty structured fields for GEN-006 that may have been
# available to the formatter or explanation generator.
excluded_columns = {
    "retrieved_context",
    "retrieved_guidance_json",
    "analyst_guidance_text",
    "final_explanation",
    "source_references_json",
    "analyst_guidance_items_json",
    "rule_trace",
}

structured_fact_records = []

for column, value in diagnostic_generation_row.items():
    if column in excluded_columns:
        continue

    if pd.isna(value):
        continue

    text_value = str(value).strip()

    if not text_value or text_value in {"[]", "{}", "None", "nan"}:
        continue

    structured_fact_records.append(
        {
            "field_name": column,
            "field_value": text_value,
        }
    )

structured_fact_frame = pd.DataFrame(structured_fact_records)

display(structured_fact_frame)

,field_name,field_value
0,evaluation_case_id,GEN-006
1,claim_id,CLM-000122
2,workflow_name,langgraph_guarded_claim_triage
3,workflow_version,langgraph_v6
4,workflow_status,COMPLETED
5,proposal_source,CUSTOM
6,deterministic_outcome,INFO_REQUIRED
7,triggering_rule_id,EVD-001
8,precedence_stage,5
9,deterministic_reason,Required evidence is missing or unreadable. Re...


In [31]:
# Inspect selected high-value fields likely to support the explanation.
support_candidate_columns = [
    column
    for column in [
        "claim_id",
        "deterministic_outcome",
        "triggering_rule_id",
        "precedence_stage",
        "deterministic_reason",
        "policy_lookup_status",
        "policy_incident_status",
        "device_match_status",
        "coverage_lookup_status",
        "covered_flag",
        "evidence_profile_id",
        "evidence_assessment_status",
        "missing_required_evidence_codes",
        "unreadable_required_evidence_codes",
        "manual_review_reason_codes",
        "follow_up_required",
        "follow_up_selection_status",
        "follow_up_question_ids",
        "projected_triage_outcome",
        "projected_triggering_rule_id",
        "projected_precedence_stage",
        "content_safety_status",
        "authority_guardrail_status",
    ]
    if column in generation_cases.columns
]

display(
    generation_cases.loc[
        generation_cases["evaluation_case_id"].eq(diagnostic_case_id),
        support_candidate_columns,
    ].T.rename(columns={generation_cases.index[
        generation_cases["evaluation_case_id"].eq(diagnostic_case_id)
    ][0]: "value"})
)

,value
claim_id,CLM-000122
deterministic_outcome,INFO_REQUIRED
triggering_rule_id,EVD-001
precedence_stage,5
deterministic_reason,Required evidence is missing or unreadable. Re...
device_match_status,DEVICE_MATCH
coverage_lookup_status,UNIQUE_COVERAGE_RECORD
covered_flag,True
evidence_profile_id,EVD-MECHANICAL-01
evidence_assessment_status,INCOMPLETE


## 17. Build Complete Authoritative Support Context

The compact support context omitted several structured facts used in the frozen explanation.

This section creates a fuller support context from a fixed whitelist of committed authoritative fields. It excludes generated and final-response fields, so the evaluated response is not used as its own evidence.

In [32]:
AUTHORITATIVE_SUPPORT_FIELDS = [
    "claim_id",
    "deterministic_outcome",
    "triggering_rule_id",
    "precedence_stage",
    "deterministic_reason",
    "decision_support_only",
    "system_limitations",
    "projected_triage_outcome",
    "projected_triggering_rule_id",
    "projected_precedence_stage",
    "claim_category",
    "requested_service_type",
    "plan_configuration_status",
    "product_scope_status",
    "coverage_lookup_status",
    "covered_flag",
    "evidence_profile_id",
    "evidence_assessment_status",
    "missing_required_evidence_codes",
    "unreadable_required_evidence_codes",
    "manual_review_reason_codes",
    "device_match_status",
    "follow_up_required",
    "follow_up_selection_status",
    "follow_up_question_ids",
    "rule_trace",
]

available_support_fields = [
    field
    for field in AUTHORITATIVE_SUPPORT_FIELDS
    if field in generation_cases.columns
]

print(
    "Authoritative support fields available:",
    len(available_support_fields),
)

for field in available_support_fields:
    print("-", field)

Authoritative support fields available: 26
- claim_id
- deterministic_outcome
- triggering_rule_id
- precedence_stage
- deterministic_reason
- decision_support_only
- system_limitations
- projected_triage_outcome
- projected_triggering_rule_id
- projected_precedence_stage
- claim_category
- requested_service_type
- plan_configuration_status
- product_scope_status
- coverage_lookup_status
- covered_flag
- evidence_profile_id
- evidence_assessment_status
- missing_required_evidence_codes
- unreadable_required_evidence_codes
- manual_review_reason_codes
- device_match_status
- follow_up_required
- follow_up_selection_status
- follow_up_question_ids
- rule_trace


In [33]:
def normalise_support_value(value: object) -> str:
    """Convert committed structured values into stable readable text."""
    if pd.isna(value):
        return "NOT_AVAILABLE"

    text = str(value).strip()

    if not text:
        return "NOT_AVAILABLE"

    return text


def build_complete_authoritative_context(
    generation_row: pd.Series,
) -> str:
    """Build support context only from whitelisted authoritative fields."""
    lines = [
        "Authoritative structured claim facts used for decision support:"
    ]

    for field in available_support_fields:
        value = normalise_support_value(
            generation_row[field]
        )

        lines.append(
            f"{field}: {value}"
        )

    return "\n".join(lines)


complete_support_context_by_case = {
    row["evaluation_case_id"]: build_complete_authoritative_context(
        row
    )
    for _, row in generation_cases.iterrows()
}

ragas_input["complete_authoritative_context"] = (
    ragas_input["evaluation_case_id"].map(
        complete_support_context_by_case
    )
)

ragas_input["complete_supporting_contexts"] = (
    ragas_input.apply(
        lambda row: (
            [row["complete_authoritative_context"]]
            + row["retrieved_contexts"]
        ),
        axis=1,
    )
)

assert ragas_input[
    "complete_authoritative_context"
].notna().all()

assert ragas_input[
    "complete_supporting_contexts"
].apply(len).eq(4).all()

assert not ragas_input[
    "complete_authoritative_context"
].str.contains(
    "generated_explanation|final_explanation",
    regex=True,
).any()

print("PASS: complete authoritative contexts created")
print("PASS: generated response fields were not used")
print("PASS: each case has one structured context plus three KB contexts")

PASS: complete authoritative contexts created
PASS: generated response fields were not used
PASS: each case has one structured context plus three KB contexts


In [34]:
diagnostic_complete_context = ragas_input.loc[
    ragas_input["evaluation_case_id"].eq("GEN-006"),
    "complete_authoritative_context",
].iloc[0]

print(diagnostic_complete_context)

Authoritative structured claim facts used for decision support:
claim_id: CLM-000122
deterministic_outcome: INFO_REQUIRED
triggering_rule_id: EVD-001
precedence_stage: 5
deterministic_reason: Required evidence is missing or unreadable. Request only the specific evidence items identified in the rule trace.
decision_support_only: True
system_limitations: ["DEV-003 is not evaluated because the runtime context does not provide a verified NOT_ENROLLED device decision.", "EXC-001 and EXC-002 are not evaluated because no structured exclusion-status dataset is available.", "Claim-history source completeness cannot be independently verified from the current runtime package."]
projected_triage_outcome: INFO_REQUIRED
projected_triggering_rule_id: EVD-001
projected_precedence_stage: 5
claim_category: MECHANICAL_BREAKDOWN
requested_service_type: REPAIR
plan_configuration_status: ACTIVE_CONFIGURATION_AVAILABLE
product_scope_status: IN_SCOPE
coverage_lookup_status: UNIQUE_COVERAGE_RECORD
covered_flag

## 18. Complete-Context Faithfulness Smoke Test

Run a final one-case Faithfulness test using the complete authoritative structured context plus the three retrieved KB chunks.

This test determines whether the earlier low score was caused by incomplete evaluator context.

In [35]:
complete_response_quality_samples = [
    SingleTurnSample(
        user_input=row.user_input,
        retrieved_contexts=row.complete_supporting_contexts,
        response=row.response,
        reference=row.rag_guidance_reference,
    )
    for row in ragas_input.itertuples(index=False)
]

complete_smoke_test_dataset = EvaluationDataset(
    samples=[
        complete_response_quality_samples[
            smoke_test_index
        ]
    ]
)

complete_smoke_test_result = evaluate(
    dataset=complete_smoke_test_dataset,
    metrics=[faithfulness],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RAGAS_RUN_CONFIG,
    raise_exceptions=False,
    show_progress=True,
)

print("PASS: complete-context faithfulness smoke test completed")

Evaluating: 100%|██████████| 1/1 [00:06<00:00,  6.71s/it]

PASS: complete-context faithfulness smoke test completed


In [36]:
complete_smoke_test_frame = (
    complete_smoke_test_result.to_pandas()
)

assert "faithfulness" in complete_smoke_test_frame.columns

complete_faithfulness = float(
    complete_smoke_test_frame.loc[
        0,
        "faithfulness",
    ]
)

assert 0.0 <= complete_faithfulness <= 1.0

faithfulness_comparison = pd.DataFrame(
    [
        {
            "evaluation_configuration": "KB_CONTEXT_ONLY",
            "faithfulness": original_faithfulness,
        },
        {
            "evaluation_configuration": (
                "COMPACT_STRUCTURED_PLUS_KB"
            ),
            "faithfulness": adjusted_faithfulness,
        },
        {
            "evaluation_configuration": (
                "COMPLETE_AUTHORITATIVE_PLUS_KB"
            ),
            "faithfulness": complete_faithfulness,
        },
    ]
)

display(faithfulness_comparison)

print(
    "Complete-context faithfulness:",
    round(complete_faithfulness, 6),
)

,evaluation_configuration,faithfulness
0,KB_CONTEXT_ONLY,0.222222
1,COMPACT_STRUCTURED_PLUS_KB,0.200000
2,COMPLETE_AUTHORITATIVE_PLUS_KB,0.777778


Complete-context faithfulness: 0.777778


## 19. Final Automated Evaluation Configuration

The smoke tests confirm that the full generated explanation must be evaluated against all authoritative inputs used during generation.

The final evaluation therefore uses two configurations:

### Retrieval-quality evaluation

- controlled retrieval query,
- three retrieved KB contexts,
- rule-level RAG guidance reference.

Metrics:

- Context Precision
- Context Recall

### Response-quality evaluation

- controlled query,
- complete authoritative structured context,
- three retrieved KB contexts,
- frozen generated explanation.

Metrics:

- Faithfulness
- Answer Relevancy

No workflow outputs are regenerated.

In [37]:
retrieval_quality_metrics = [
    context_precision,
    context_recall,
]

response_quality_metrics = [
    faithfulness,
    answer_relevancy,
]

complete_response_quality_dataset = EvaluationDataset(
    samples=complete_response_quality_samples
)

assert len(retrieval_quality_dataset) == EXPECTED_CASE_COUNT
assert len(complete_response_quality_dataset) == EXPECTED_CASE_COUNT

assert all(
    len(sample.retrieved_contexts) == 3
    for sample in retrieval_quality_dataset
)

assert all(
    len(sample.retrieved_contexts) == 4
    for sample in complete_response_quality_dataset
)

print("PASS: final retrieval-quality configuration validated")
print("PASS: final response-quality configuration validated")
print(
    "Retrieval-quality metrics:",
    [metric.name for metric in retrieval_quality_metrics],
)
print(
    "Response-quality metrics:",
    [metric.name for metric in response_quality_metrics],
)

PASS: final retrieval-quality configuration validated
PASS: final response-quality configuration validated
Retrieval-quality metrics: ['context_precision', 'context_recall']
Response-quality metrics: ['faithfulness', 'answer_relevancy']


## 20. Run Retrieval-Quality Evaluation

Evaluate whether the actual workflow retrieval supplied useful and sufficiently complete KB guidance for the 12 frozen development cases.

In [38]:
retrieval_quality_result = evaluate(
    dataset=retrieval_quality_dataset,
    metrics=retrieval_quality_metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RAGAS_RUN_CONFIG,
    raise_exceptions=False,
    show_progress=True,
)

print("PASS: retrieval-quality evaluation completed")

Evaluating: 100%|██████████| 24/24 [01:15<00:00,  3.13s/it]

PASS: retrieval-quality evaluation completed


In [39]:
retrieval_quality_frame = (
    retrieval_quality_result.to_pandas()
)

required_retrieval_metrics = {
    "context_precision",
    "context_recall",
}

missing_retrieval_metrics = (
    required_retrieval_metrics
    - set(retrieval_quality_frame.columns)
)

assert not missing_retrieval_metrics, (
    "Missing retrieval metrics: "
    + ", ".join(sorted(missing_retrieval_metrics))
)

assert retrieval_quality_frame[
    sorted(required_retrieval_metrics)
].notna().all().all()

assert retrieval_quality_frame[
    sorted(required_retrieval_metrics)
].apply(
    lambda series: series.between(0.0, 1.0).all()
).all()

print("PASS: all retrieval-quality scores are present")
print("PASS: all retrieval-quality scores are within 0–1")

display(
    retrieval_quality_frame[
        [
            "context_precision",
            "context_recall",
        ]
    ]
)

PASS: all retrieval-quality scores are present
PASS: all retrieval-quality scores are within 0–1


,context_precision,context_recall
0,0.500000,0.0
1,0.500000,0.0
2,0.000000,0.0
3,0.500000,0.0
4,1.000000,1.0
5,0.583333,1.0
6,1.000000,1.0
7,0.500000,0.5
8,0.333333,0.0
9,1.000000,1.0


## 21. Run Response-Quality Evaluation

Evaluate the frozen explanation against the complete authoritative support context used by the hybrid workflow.

In [40]:
response_quality_result = evaluate(
    dataset=complete_response_quality_dataset,
    metrics=response_quality_metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RAGAS_RUN_CONFIG,
    raise_exceptions=False,
    show_progress=True,
)

print("PASS: response-quality evaluation completed")

Evaluating: 100%|██████████| 24/24 [02:00<00:00,  5.01s/it]

PASS: response-quality evaluation completed


In [41]:
response_quality_frame = (
    response_quality_result.to_pandas()
)

required_response_metrics = {
    "faithfulness",
    "answer_relevancy",
}

missing_response_metrics = (
    required_response_metrics
    - set(response_quality_frame.columns)
)

assert not missing_response_metrics, (
    "Missing response metrics: "
    + ", ".join(sorted(missing_response_metrics))
)

assert response_quality_frame[
    sorted(required_response_metrics)
].notna().all().all()

assert response_quality_frame[
    sorted(required_response_metrics)
].apply(
    lambda series: series.between(0.0, 1.0).all()
).all()

print("PASS: all response-quality scores are present")
print("PASS: all response-quality scores are within 0–1")

display(
    response_quality_frame[
        [
            "faithfulness",
            "answer_relevancy",
        ]
    ]
)

PASS: all response-quality scores are present
PASS: all response-quality scores are within 0–1


,faithfulness,answer_relevancy
0,0.600000,0.467954
1,0.384615,0.485018
2,0.750000,0.476625
3,0.600000,0.506590
4,0.857143,0.519879
5,0.800000,0.543517
6,0.500000,0.549214
7,0.562500,0.586636
8,0.666667,0.602571
9,0.600000,0.563822


## 22. Consolidate Automated Evaluation Results

Combine the four Ragas metrics with case IDs, deterministic rules, human-review scores, and the deterministic reference-chunk diagnostic.

In [42]:
automated_results = ragas_input[
    [
        "evaluation_case_id",
        "claim_id",
        "deterministic_outcome",
        "triggering_rule_id",
        "precedence_stage",
        "reference_chunk_hit",
        "reference_chunks_retrieved",
    ]
].reset_index(drop=True).copy()

automated_results["context_precision"] = (
    retrieval_quality_frame["context_precision"].values
)

automated_results["context_recall"] = (
    retrieval_quality_frame["context_recall"].values
)

automated_results["faithfulness"] = (
    response_quality_frame["faithfulness"].values
)

automated_results["answer_relevancy"] = (
    response_quality_frame["answer_relevancy"].values
)

human_metric_columns = [
    "evaluation_case_id",
    "context_relevance_human",
    "groundedness_human",
    "answer_relevance_human",
    "hallucination_control_human",
    "critical_safety_failure_human",
]

automated_results = automated_results.merge(
    human_review[human_metric_columns],
    on="evaluation_case_id",
    how="left",
    validate="one_to_one",
)

assert len(automated_results) == EXPECTED_CASE_COUNT
assert automated_results["evaluation_case_id"].nunique() == EXPECTED_CASE_COUNT

automated_metric_columns = [
    "context_precision",
    "context_recall",
    "faithfulness",
    "answer_relevancy",
]

assert automated_results[
    automated_metric_columns
].notna().all().all()

print("PASS: 12 automated-evaluation records consolidated")
print("PASS: automated and human evidence joined")

display(automated_results)

PASS: 12 automated-evaluation records consolidated
PASS: automated and human evidence joined


,evaluation_case_id,claim_id,deterministic_outcome,triggering_rule_id,precedence_stage,reference_chunk_hit,reference_chunks_retrieved,context_precision,context_recall,faithfulness,answer_relevancy,context_relevance_human,groundedness_human,answer_relevance_human,hallucination_control_human,critical_safety_failure_human
0,GEN-001,CLM-000001,PROCEED,OUT-001,6,False,[],0.500000,0.0,0.600000,0.467954,3,4,4,4,NO
1,GEN-002,CLM-000053,PROCEED,OUT-001,6,False,[],0.500000,0.0,0.384615,0.485018,3,4,4,4,NO
2,GEN-003,CLM-000177,PROCEED,OUT-001,6,False,[],0.000000,0.0,0.750000,0.476625,3,4,4,4,NO
3,GEN-004,CLM-000103,INFO_REQUIRED,DEV-001,3,False,[],0.500000,0.0,0.600000,0.506590,2,4,4,4,NO
4,GEN-005,CLM-000104,INFO_REQUIRED,DEV-001,3,False,[],1.000000,1.0,0.857143,0.519879,2,4,4,4,NO
5,GEN-006,CLM-000122,INFO_REQUIRED,EVD-001,5,True,[KB-002::S02],0.583333,1.0,0.800000,0.543517,4,4,4,4,NO
6,GEN-007,CLM-000149,MANUAL_REVIEW,EVD-002,1,True,[KB-002::S02],1.000000,1.0,0.500000,0.549214,4,4,4,4,NO
7,GEN-008,CLM-000164,MANUAL_REVIEW,DATA-002,1,False,[],0.500000,0.5,0.562500,0.586636,2,4,4,4,NO
8,GEN-009,CLM-000180,MANUAL_REVIEW,ANM-001,4,False,[],0.333333,0.0,0.666667,0.602571,2,4,3,4,NO
9,GEN-010,CLM-000154,NOT_ELIGIBLE,COV-001,2,True,[KB-002::S02],1.000000,1.0,0.600000,0.563822,3,3,3,3,NO


## 23. Overall Automated Evaluation Summary

Summarise the four automated RAG evaluation metrics across all twelve frozen development cases.

These statistics provide the primary quantitative evidence reported in the final project documentation.

In [43]:
automated_summary = pd.DataFrame(
    [
        {
            "metric": "Context Precision",
            "mean": automated_results["context_precision"].mean(),
            "median": automated_results["context_precision"].median(),
            "minimum": automated_results["context_precision"].min(),
            "maximum": automated_results["context_precision"].max(),
        },
        {
            "metric": "Context Recall",
            "mean": automated_results["context_recall"].mean(),
            "median": automated_results["context_recall"].median(),
            "minimum": automated_results["context_recall"].min(),
            "maximum": automated_results["context_recall"].max(),
        },
        {
            "metric": "Faithfulness",
            "mean": automated_results["faithfulness"].mean(),
            "median": automated_results["faithfulness"].median(),
            "minimum": automated_results["faithfulness"].min(),
            "maximum": automated_results["faithfulness"].max(),
        },
        {
            "metric": "Answer Relevancy",
            "mean": automated_results["answer_relevancy"].mean(),
            "median": automated_results["answer_relevancy"].median(),
            "minimum": automated_results["answer_relevancy"].min(),
            "maximum": automated_results["answer_relevancy"].max(),
        },
    ]
)

display(
    automated_summary.round(3)
)

print("PASS: overall summary metrics calculated")

,metric,mean,median,minimum,maximum
0,Context Precision,0.576,0.50,0.000,1.000
1,Context Recall,0.417,0.25,0.000,1.000
2,Faithfulness,0.627,0.60,0.385,0.857
3,Answer Relevancy,0.533,0.54,0.468,0.603


PASS: overall summary metrics calculated


## 24. Performance by Deterministic Rule

Analyse automated evaluation scores by deterministic routing rule.

This identifies which workflow decisions are well supported by retrieved guidance and which rule families represent future retrieval-improvement opportunities.

In [44]:
rule_level_summary = (
    automated_results
    .groupby(
        [
            "triggering_rule_id",
            "deterministic_outcome",
        ],
        dropna=False,
    )
    .agg(
        cases=("evaluation_case_id", "count"),
        context_precision=("context_precision", "mean"),
        context_recall=("context_recall", "mean"),
        faithfulness=("faithfulness", "mean"),
        answer_relevancy=("answer_relevancy", "mean"),
        reference_chunk_hit_rate=(
            "reference_chunk_hit",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "deterministic_outcome",
            "triggering_rule_id",
        ]
    )
)

display(
    rule_level_summary.round(3)
)

print("PASS: rule-level summary calculated")

,triggering_rule_id,deterministic_outcome,cases,context_precision,context_recall,faithfulness,answer_relevancy,reference_chunk_hit_rate
3,DEV-001,INFO_REQUIRED,2,0.750,0.50,0.729,0.513,0.0
4,EVD-001,INFO_REQUIRED,1,0.583,1.00,0.800,0.544,1.0
0,ANM-001,MANUAL_REVIEW,1,0.333,0.00,0.667,0.603,0.0
2,DATA-002,MANUAL_REVIEW,1,0.500,0.50,0.562,0.587,0.0
5,EVD-002,MANUAL_REVIEW,1,1.000,1.00,0.500,0.549,1.0
1,COV-001,NOT_ELIGIBLE,2,1.000,0.75,0.550,0.561,0.5
6,LIM-001,NOT_ELIGIBLE,1,0.000,0.00,0.700,0.537,0.0
7,OUT-001,PROCEED,3,0.333,0.00,0.578,0.477,0.0


PASS: rule-level summary calculated


## 25. Automated and Human Evaluation Comparison

Compare automated RAG metrics with the independent human review completed during Notebook 07.

The two evaluations measure different properties and therefore complement rather than replace each other.

In [46]:
comparison_summary = pd.DataFrame(
    [
        {
            "Evaluation": "Automated RAG Metrics",
            "Primary Focus": (
                "Retrieval quality and explanation support"
            ),
        },
        {
            "Evaluation": "Human Review",
            "Primary Focus": (
                "Analyst usefulness, safety, and groundedness"
            ),
        },
    ]
)

display(comparison_summary)

print("Average Human Scores")

human_score_summary = pd.DataFrame(
    {
        "Metric": [
            "Context relevance",
            "Groundedness",
            "Answer relevance",
            "Hallucination control",
        ],
        "Average": [
            human_review["context_relevance_human"].mean(),
            human_review["groundedness_human"].mean(),
            human_review["answer_relevance_human"].mean(),
            human_review["hallucination_control_human"].mean(),
        ],
    }
)

display(human_score_summary.round(2))

critical_safety_failures = (
    human_review["critical_safety_failure_human"]
    .astype(str)
    .str.upper()
    .eq("YES")
    .sum()
)

print(
    "Critical safety failures:",
    int(critical_safety_failures),
)

assert len(comparison_summary) == 2
assert len(human_score_summary) == 4
assert critical_safety_failures == 0

print("PASS: automated and human evaluation roles documented")
print("PASS: human-review summary calculated")
print("PASS: no critical human-reviewed safety failures")

,Evaluation,Primary Focus
0,Automated RAG Metrics,Retrieval quality and explanation support
1,Human Review,"Analyst usefulness, safety, and groundedness"


Average Human Scores


,Metric,Average
0,Context relevance,2.75
1,Groundedness,3.75
2,Answer relevance,3.67
3,Hallucination control,3.75


Critical safety failures: 0
PASS: automated and human evaluation roles documented
PASS: human-review summary calculated
PASS: no critical human-reviewed safety failures


## 26. Low-Score and Retrieval-Gap Review

Review the lowest-scoring cases without treating Ragas scores as formal pass/fail thresholds.

For each automated metric, the three lowest cases are selected for qualitative review. A case may appear under more than one metric.

In [47]:
# Select the three lowest cases for each automated metric.

LOW_SCORE_REVIEW_COUNT = 3

review_metric_columns = [
    "context_precision",
    "context_recall",
    "faithfulness",
    "answer_relevancy",
]

low_score_frames = []

for metric_name in review_metric_columns:
    metric_review = (
        automated_results
        .nsmallest(
            LOW_SCORE_REVIEW_COUNT,
            metric_name,
            keep="all",
        )
        [
            [
                "evaluation_case_id",
                "claim_id",
                "deterministic_outcome",
                "triggering_rule_id",
                "reference_chunk_hit",
                "reference_chunks_retrieved",
                metric_name,
            ]
        ]
        .copy()
    )

    metric_review.insert(
        0,
        "review_metric",
        metric_name,
    )

    metric_review = metric_review.rename(
        columns={
            metric_name: "metric_score",
        }
    )

    low_score_frames.append(metric_review)

low_score_review = (
    pd.concat(
        low_score_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "review_metric",
            "metric_score",
            "evaluation_case_id",
        ]
    )
    .reset_index(drop=True)
)

assert set(low_score_review["review_metric"]) == set(
    review_metric_columns
)

print("PASS: low-score review table created")

display(low_score_review)

PASS: low-score review table created


,review_metric,evaluation_case_id,claim_id,deterministic_outcome,triggering_rule_id,reference_chunk_hit,reference_chunks_retrieved,metric_score
0,answer_relevancy,GEN-001,CLM-000001,PROCEED,OUT-001,False,[],0.467954
1,answer_relevancy,GEN-003,CLM-000177,PROCEED,OUT-001,False,[],0.476625
2,answer_relevancy,GEN-002,CLM-000053,PROCEED,OUT-001,False,[],0.485018
3,context_precision,GEN-003,CLM-000177,PROCEED,OUT-001,False,[],0.000000
4,context_precision,GEN-012,CLM-000206,NOT_ELIGIBLE,LIM-001,False,[],0.000000
5,context_precision,GEN-009,CLM-000180,MANUAL_REVIEW,ANM-001,False,[],0.333333
6,context_recall,GEN-001,CLM-000001,PROCEED,OUT-001,False,[],0.000000
7,context_recall,GEN-002,CLM-000053,PROCEED,OUT-001,False,[],0.000000
8,context_recall,GEN-003,CLM-000177,PROCEED,OUT-001,False,[],0.000000
9,context_recall,GEN-004,CLM-000103,INFO_REQUIRED,DEV-001,False,[],0.000000


## 27. Exact-Chunk and Semantic-Coverage Comparison

The deterministic provenance diagnostic checks whether a manually identified ideal KB chunk was retrieved.

Ragas Context Recall instead evaluates whether the necessary guidance was semantically present in any retrieved context. The two measures are therefore complementary.

In [48]:
# Compare exact reference-chunk hits with semantic context coverage.

automated_results["semantic_context_coverage"] = (
    automated_results["context_recall"] > 0.0
)

exact_chunk_hit_count = int(
    automated_results["reference_chunk_hit"].sum()
)

semantic_coverage_count = int(
    automated_results["semantic_context_coverage"].sum()
)

semantic_without_exact_hit_count = int(
    (
        automated_results["semantic_context_coverage"]
        & ~automated_results["reference_chunk_hit"]
    ).sum()
)

retrieval_diagnostic_summary = pd.DataFrame(
    [
        {
            "diagnostic": "Exact preferred chunk retrieved",
            "case_count": exact_chunk_hit_count,
            "case_rate": (
                exact_chunk_hit_count
                / EXPECTED_CASE_COUNT
            ),
        },
        {
            "diagnostic": "Some reference guidance semantically covered",
            "case_count": semantic_coverage_count,
            "case_rate": (
                semantic_coverage_count
                / EXPECTED_CASE_COUNT
            ),
        },
        {
            "diagnostic": (
                "Semantic coverage without exact preferred chunk"
            ),
            "case_count": semantic_without_exact_hit_count,
            "case_rate": (
                semantic_without_exact_hit_count
                / EXPECTED_CASE_COUNT
            ),
        },
    ]
)

display(
    retrieval_diagnostic_summary.assign(
        case_rate=lambda frame: (
            frame["case_rate"]
            .mul(100)
            .round(1)
            .astype(str)
            + "%"
        )
    )
)

print("PASS: provenance and semantic diagnostics compared")

,diagnostic,case_count,case_rate
0,Exact preferred chunk retrieved,3,25.0%
1,Some reference guidance semantically covered,6,50.0%
2,Semantic coverage without exact preferred chunk,3,25.0%


PASS: provenance and semantic diagnostics compared


## 28. Export Automated Evaluation Artifacts

Export case-level results, overall summaries, rule-level summaries, low-score review records, and a reproducibility manifest.

All outputs are based on the twelve frozen development cases. No held-out claim is included, and no workflow response is regenerated.

In [49]:
# Export reproducible Notebook 09 evaluation artifacts.

from datetime import datetime, timezone
from pathlib import Path
import json
import ragas

RAGAS_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "ragas"
)

RAGAS_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

case_results_path = (
    RAGAS_OUTPUT_DIR
    / "ragas_case_results_v1.csv"
)

summary_path = (
    RAGAS_OUTPUT_DIR
    / "ragas_summary_v1.csv"
)

rule_summary_path = (
    RAGAS_OUTPUT_DIR
    / "ragas_rule_summary_v1.csv"
)

low_score_review_path = (
    RAGAS_OUTPUT_DIR
    / "ragas_low_score_review_v1.csv"
)

manifest_path = (
    RAGAS_OUTPUT_DIR
    / "ragas_manifest_v1.json"
)

automated_results.to_csv(
    case_results_path,
    index=False,
)

automated_summary.to_csv(
    summary_path,
    index=False,
)

rule_level_summary.to_csv(
    rule_summary_path,
    index=False,
)

low_score_review.to_csv(
    low_score_review_path,
    index=False,
)

metric_summary_for_manifest = {
    row["metric"]: {
        "mean": round(float(row["mean"]), 6),
        "median": round(float(row["median"]), 6),
        "minimum": round(float(row["minimum"]), 6),
        "maximum": round(float(row["maximum"]), 6),
    }
    for _, row in automated_summary.iterrows()
}

human_summary_for_manifest = {
    "context_relevance_mean": round(
        float(
            human_review[
                "context_relevance_human"
            ].mean()
        ),
        6,
    ),
    "groundedness_mean": round(
        float(
            human_review[
                "groundedness_human"
            ].mean()
        ),
        6,
    ),
    "answer_relevance_mean": round(
        float(
            human_review[
                "answer_relevance_human"
            ].mean()
        ),
        6,
    ),
    "hallucination_control_mean": round(
        float(
            human_review[
                "hallucination_control_human"
            ].mean()
        ),
        6,
    ),
    "critical_safety_failures": int(
        critical_safety_failures
    ),
}

ragas_manifest = {
    "artifact_name": "Automated RAG Evaluation",
    "artifact_version": "v1",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "notebooks/"
        "09_automated_rag_evaluation.ipynb"
    ),
    "framework": {
        "name": "Ragas",
        "version": ragas.__version__,
    },
    "evaluator_configuration": {
        "llm": "gpt-5.4-mini",
        "embedding_model": (
            "text-embedding-3-small"
        ),
        "timeout_seconds": 120,
        "max_retries": 2,
        "max_workers": 1,
    },
    "evaluation_scope": {
        "case_count": int(
            EXPECTED_CASE_COUNT
        ),
        "case_partition": (
            "frozen development generation cases"
        ),
        "held_out_data_used": False,
        "workflow_outputs_regenerated": False,
    },
    "methodology": {
        "retrieval_quality": {
            "contexts": (
                "three retrieved approved-KB chunks"
            ),
            "reference": (
                "rule-level RAG guidance reference"
            ),
            "metrics": [
                "context_precision",
                "context_recall",
            ],
        },
        "response_quality": {
            "contexts": (
                "complete authoritative structured "
                "facts plus three retrieved approved-KB "
                "chunks"
            ),
            "response": (
                "frozen generated explanation"
            ),
            "metrics": [
                "faithfulness",
                "answer_relevancy",
            ],
        },
        "hybrid_context_rationale": (
            "The generated explanation combines "
            "deterministic structured facts and "
            "retrieved KB guidance. Faithfulness was "
            "therefore evaluated against all "
            "authoritative inputs available during "
            "generation."
        ),
    },
    "metric_summary": metric_summary_for_manifest,
    "human_review_summary": (
        human_summary_for_manifest
    ),
    "retrieval_diagnostics": {
        "exact_reference_chunk_hits": (
            exact_chunk_hit_count
        ),
        "semantic_context_coverage_cases": (
            semantic_coverage_count
        ),
        "semantic_coverage_without_exact_hit": (
            semantic_without_exact_hit_count
        ),
    },
    "source_artifacts": [
        (
            "data/evaluation/generation/"
            "generation_evaluation_cases_v1.csv"
        ),
        (
            "data/evaluation/generation/"
            "generation_human_review_v2.csv"
        ),
        (
            "data/evaluation/generation/"
            "generation_judge_input_v2.csv"
        ),
        (
            "data/evaluation/generation/"
            "generation_llm_judge_results_v2.csv"
        ),
        (
            "data/evaluation/generation/"
            "generation_calibration_summary_v2.csv"
        ),
    ],
    "output_artifacts": [
        case_results_path.name,
        summary_path.name,
        rule_summary_path.name,
        low_score_review_path.name,
        manifest_path.name,
    ],
}

with manifest_path.open(
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        ragas_manifest,
        manifest_file,
        indent=2,
    )

print("PASS: Ragas evaluation artifacts exported")

for output_path in [
    case_results_path,
    summary_path,
    rule_summary_path,
    low_score_review_path,
    manifest_path,
]:
    print(
        "-",
        output_path.relative_to(
            PROJECT_ROOT
        ),
    )

PASS: Ragas evaluation artifacts exported
- data/evaluation/ragas/ragas_case_results_v1.csv
- data/evaluation/ragas/ragas_summary_v1.csv
- data/evaluation/ragas/ragas_rule_summary_v1.csv
- data/evaluation/ragas/ragas_low_score_review_v1.csv
- data/evaluation/ragas/ragas_manifest_v1.json


In [50]:
# Validate that all exported artifacts exist and are non-empty.

exported_paths = [
    case_results_path,
    summary_path,
    rule_summary_path,
    low_score_review_path,
    manifest_path,
]

for exported_path in exported_paths:
    assert exported_path.exists(), (
        f"Missing export: {exported_path}"
    )

    assert exported_path.stat().st_size > 0, (
        f"Empty export: {exported_path}"
    )

exported_case_results = pd.read_csv(
    case_results_path
)

exported_summary = pd.read_csv(
    summary_path
)

exported_rule_summary = pd.read_csv(
    rule_summary_path
)

with manifest_path.open(
    "r",
    encoding="utf-8",
) as manifest_file:
    exported_manifest = json.load(
        manifest_file
    )

assert len(exported_case_results) == EXPECTED_CASE_COUNT
assert len(exported_summary) == 4
assert not exported_rule_summary.empty
assert (
    exported_manifest[
        "evaluation_scope"
    ]["held_out_data_used"]
    is False
)

print("PASS: all exported artifacts exist and are non-empty")
print("PASS: exported case count validated")
print("PASS: four metric summaries validated")
print("PASS: held-out exclusion recorded in manifest")

PASS: all exported artifacts exist and are non-empty
PASS: exported case count validated
PASS: four metric summaries validated
PASS: held-out exclusion recorded in manifest


## 29. Conclusions

1. The automated evaluation covered all 12 frozen development cases without regenerating workflow outputs or accessing the held-out partition.

2. Mean automated scores were:
   - Context Precision: **0.576**
   - Context Recall: **0.417**
   - Faithfulness: **0.627**
   - Answer Relevancy: **0.533**

3. Retrieval alignment was the primary weakness. Several controlled queries retrieved generic evidence guidance rather than guidance specific to the triggering deterministic rule.

4. Exact preferred reference chunks were retrieved for 3 of 12 cases. Ragas identified semantic reference coverage in 6 of 12 cases, showing that useful guidance was sometimes present in an alternative chunk.

5. The original KB-only Faithfulness smoke test underestimated response support because the workflow also supplies authoritative structured facts to the explanation generator.

6. Using the complete authoritative support context increased the `GEN-006` Faithfulness score from **0.222** to **0.778**, validating the hybrid evaluation configuration.

7. Across all 12 cases, mean Faithfulness was **0.627**. The result indicates moderate explanation support while also identifying individual cases requiring qualitative review.

8. Answer Relevancy was moderate and narrowly distributed. This metric should be interpreted cautiously because the input is a structured controlled query rather than a conventional user question.

9. Human review remained stronger on practical analyst usefulness, groundedness, relevance, and hallucination control, with no critical safety failures.

10. Automated and human evaluations are complementary: Ragas exposes retrieval and support gaps, while human review assesses operational usefulness, authority boundaries, and safety.

11. No deterministic outcome or triggering rule was changed by RAG, the explanation model, the automated evaluator, or the human-review process.

12. The results support retaining Semantic Embedding as the default retrieval method, with the cross-encoder reranker as a controlled optional stage. Rule-aware query alignment remains the principal future improvement opportunity.